# Lesson 04 - ツール使用デザインパターン

このレッスンでは、Microsoft Agent Framework（Python）を使ったAIエージェントの**ツール使用**デザインパターンを学びます。内容は次の通りです：

- `@tool`デコレーターと型付きパラメータで関数ツールを定義する方法
- モデルが各ツールの機能を理解できるようツールスキーマを提供する方法
- `approval_mode`でツールの実行を制御する方法
- Pydanticモデルと`response_format`を使って**構造化された出力**を返す方法

シナリオは、目的地の検索、空席確認、フライト情報取得ができる**旅行予約エージェント**です。


## セットアップ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @toolデコレーターによるツールの定義

`@tool`デコレーターは、単純なPython関数をエージェントが呼び出せるツールに変えます。  
主なポイント：

- **ドキュメンテーション文字列**はモデルが見るツールの説明になります。  
- **型アノテーション**（説明付きの`Annotated`を含む）はツールスキーマを定義します。  
- `approval_mode`は実行前にユーザーが各呼び出しを承認する必要があるかを制御します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 複数のツールを持つエージェントの作成

3つのツールすべてをクライアントに渡して、モデルがユーザーの質問に答えるために必要なものを呼び出せるようにします。


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ツールを用いた構造化出力

`response_format` を Pydantic モデルに設定することで、エージェントは自由形式のテキストではなく、型の決まった JSON オブジェクトを返すことが強制されます。これは、下流のコードが結果をプログラム的に利用する必要がある場合に有用です。


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ツール承認パターン

`@tool` の `approval_mode` パラメーターは、ツールの呼び出しが実行前に人間の承認を必要とするかどうかを制御します：

| モード | 動作 |
|---|---|
| `"never_require"` | ツールは自動的に実行されます — ユーザーの確認は不要です。 |
| `"always_require"` | 実行前に毎回ユーザーの承認が必要です。 |

副作用のあるツール（例：フライトの予約、クレジットカードの請求）には `"always_require"` を使用し、人間が介在するようにします。


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

このレッスンでは以下を学びました：

1. `@tool` デコレーターを使って型付きパラメーターとツールスキーマとして機能するドキュメンテーション文字列で **ツールを定義する方法**。
2. エージェントが複雑な問い合わせに答えるために **複数のツールを連続して呼び出せるように組み合わせる方法**。
3. Pydanticモデルを `response_format` として渡すことで **構造化された出力を返す方法**。
4. 敏感な操作に対して人間の確認を入れるために `approval_mode` で **ツール承認を制御する方法**。

これらのパターンは、外部システムと安全にやり取りできる信頼性の高い本番環境向けエージェントを構築する基盤となります。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を用いて翻訳されたものです。正確性の向上に努めておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご了承ください。原文の言語による文書が正式な情報源と見なされます。重要な情報については、専門の人間による翻訳を推奨いたします。本翻訳の利用により生じたいかなる誤解や誤訳についても、一切の責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
